In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from useful_func1 import missing_info, out_info

In [2]:
df= pd.read_csv("Loan_approval_data_2025.csv")

In [3]:
df.head(20)

,customer_id,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,loan_status
0,CUST100000,40,Employed,17.2,25579,692,5.3,895,10820,0,0,0,Credit Card,Business,600,17.02,0.423,0.023,0.008,1
1,CUST100001,33,Employed,7.3,43087,627,3.5,169,16550,0,1,0,Personal Loan,Home Improvement,53300,14.10,0.384,1.237,0.412,0
2,CUST100002,42,Student,1.1,20840,689,8.4,17,7852,0,0,0,Credit Card,Debt Consolidation,2100,18.33,0.377,0.101,0.034,1
3,CUST100003,53,Student,0.5,29147,692,9.8,1480,11603,0,1,0,Credit Card,Business,2900,18.74,0.398,0.099,0.033,1
4,CUST100004,32,Employed,12.5,63657,630,7.2,209,12424,0,0,0,Personal Loan,Education,99600,13.92,0.195,1.565,0.522,1
5,CUST100005,32,Employed,13.4,32015,570,7.3,253,1120,0,0,2,Credit Card,Personal,37000,22.92,0.035,1.156,0.385,0
6,CUST100006,53,Employed,22.9,44989,674,11.1,19667,19298,0,0,0,Personal Loan,Home Improvement,45600,11.02,0.429,1.014,0.338,1
7,CUST100007,44,Self-Employed,4.2,80603,625,18.5,830,38382,0,0,0,Credit Card,Personal,51700,19.42,0.476,0.641,0.214,1
8,CUST100008,29,Employed,5.9,28416,569,2.6,1334,22668,1,2,0,Credit Card,Education,33800,22.72,0.798,1.189,0.396,0
9,CUST100009,41,Employed,7.0,70717,638,21.5,1578,21394,0,1,0,Credit Card,Personal,70000,19.35,0.303,0.990,0.330,1


In [4]:
## drop customer_id(unwanted column)
df.drop(columns = ["customer_id"], inplace = True)

In [5]:
df.columns

Index(['age', 'occupation_status', 'years_employed', 'annual_income',
       'credit_score', 'credit_history_years', 'savings_assets',
       'current_debt', 'defaults_on_file', 'delinquencies_last_2yrs',
       'derogatory_marks', 'product_type', 'loan_intent', 'loan_amount',
       'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio',
       'payment_to_income_ratio', 'loan_status'],
      dtype='object')

In [6]:
df.isnull().sum()

age                        0
occupation_status          0
years_employed             0
annual_income              0
credit_score               0
credit_history_years       0
savings_assets             0
current_debt               0
defaults_on_file           0
delinquencies_last_2yrs    0
derogatory_marks           0
product_type               0
loan_intent                0
loan_amount                0
interest_rate              0
debt_to_income_ratio       0
loan_to_income_ratio       0
payment_to_income_ratio    0
loan_status                0
dtype: int64

In [7]:
## define X,y
X = df.drop('loan_status', axis=1)
y = df['loan_status']
print(f"shape of X-:{X.shape}")
print(f"shape of y-:{y.shape}")

shape of X-:(50000, 18)
shape of y-:(50000,)


In [8]:
from sklearn.model_selection import train_test_split as tts
X_train, X_test, y_train, y_test = tts(X,y, test_size = 0.2, random_state =42)

In [9]:
print(f"shape of X_train:- {X_train.shape}")
print(f"shape of X_test:- {X_test.shape}")
print(f"shape of y_train:- {y_train.shape}")
print(f"shape of y_test:- {y_test.shape}")

shape of X_train:- (40000, 18)
shape of X_test:- (10000, 18)
shape of y_train:- (40000,)
shape of y_test:- (10000,)


## check outliers

In [10]:
num_cols = [col for col in df.columns if df[col].dtype != "O"]

In [11]:
## function for checking outliers
def out_info(df: pd.DataFrame) -> pd.DataFrame:
    
    # Select numeric columns
    num_cols = [col for col in df.columns if df[col].dtype != "O"]

    # Compute skewness
    skew = df[num_cols].skew()
    z_score_cols = skew[abs(skew) <= 0.5].index  # approximately normal
    iqr_cols = skew[abs(skew) > 0.5].index       # skewed

    # Initialize results DataFrame
    out_df = pd.DataFrame(columns=[
        "col_name", "method", "num_outliers", "%_outliers", "uw", "lw"
    ])

    # Z-score method for approximately normal columns
    for col in z_score_cols:
        col_mean = df[col].mean()
        col_std = df[col].std()
        uw = col_mean + 3 * col_std
        lw = col_mean - 3 * col_std
        mask = (df[col] > uw) | (df[col] < lw)
        num_out = mask.sum()
        per_out = round(mask.mean() * 100, 2)
        out_df.loc[len(out_df)] = [col, "Z_Score", num_out, per_out, uw, lw]

    # IQR method for skewed columns
    for col in iqr_cols:
        q1 = df[col].quantile(0.25)
        q3 = df[col].quantile(0.75)
        iqr = q3 - q1
        uw = q3 + 1.5 * iqr
        lw = q1 - 1.5 * iqr
        mask = (df[col] > uw) | (df[col] < lw)
        num_out = mask.sum()
        per_out = round(mask.mean() * 100, 2)
        out_df.loc[len(out_df)] = [col, "IQR", num_out, per_out, uw, lw]

    # Keep only columns with outliers and sort
    out_df = out_df[out_df["num_outliers"] > 0] \
                 .sort_values(by="num_outliers", ascending=False) \
                 .reset_index(drop=True)

    return out_df


In [12]:
## check outliers
out_info(X_test)

,col_name,method,num_outliers,%_outliers,uw,lw
0,savings_assets,IQR,1309,13.09,5446.625000,-3058.375000
1,derogatory_marks,IQR,1273,12.73,0.000000,0.000000
2,current_debt,IQR,593,5.93,37883.875000,-13797.125000
3,defaults_on_file,IQR,511,5.11,0.000000,0.000000
4,annual_income,IQR,503,5.03,114658.625000,-25242.375000
5,delinquencies_last_2yrs,IQR,342,3.42,2.500000,-1.500000
6,years_employed,IQR,250,2.50,26.550000,-13.850000
7,credit_history_years,IQR,78,0.78,28.250000,-13.750000
8,debt_to_income_ratio,IQR,72,0.72,0.737500,-0.186500
9,credit_score,Z_Score,28,0.28,838.130606,449.048194


In [15]:
cols = ['annual_income', 'current_debt', 'savings_assets']

In [16]:
X_train['annual_income_log'] = np.log1p(X_train['annual_income'])
X_train['current_debt_log'] = np.log1p(X_train['current_debt'])
X_train['savings_assets_log'] = np.log1p(X_train['savings_assets'])

In [17]:
for col in cols:
    print(f"{col} skew before :", X_train[col].skew())
    print(f"{col} skew after  :", X_train[f"{col}_log"].skew())
    print("-"*40)

annual_income skew before : 1.8777989909432247
annual_income skew after  : 0.17866000343430227
----------------------------------------
current_debt skew before : 2.424658745677445
current_debt skew after  : -0.4603436258785038
----------------------------------------
savings_assets skew before : 11.741532761728665
savings_assets skew after  : -0.24789075498268465
----------------------------------------


In [18]:
X_train.head()

,age,occupation_status,years_employed,annual_income,credit_score,credit_history_years,savings_assets,current_debt,defaults_on_file,delinquencies_last_2yrs,...,product_type,loan_intent,loan_amount,interest_rate,debt_to_income_ratio,loan_to_income_ratio,payment_to_income_ratio,annual_income_log,current_debt_log,savings_assets_log
39087,22,Employed,2.3,17826,555,0.3,12,1204,0,1,...,Credit Card,Personal,1400,22.45,0.068,0.079,0.026,9.788469,7.094235,2.564949
30893,30,Student,1.1,21641,629,7.2,3099,7597,0,0,...,Personal Loan,Education,1300,14.40,0.351,0.060,0.020,9.982391,8.935640,8.039157
45278,29,Employed,9.7,70880,577,0.6,43,4363,1,1,...,Personal Loan,Personal,100000,15.86,0.062,1.411,0.470,11.168758,8.381144,3.784190
16398,45,Employed,10.0,57503,590,24.6,2214,7828,0,1,...,Line of Credit,Personal,32600,14.73,0.136,0.567,0.189,10.959610,8.965590,7.703008
13653,32,Employed,3.4,29187,693,12.0,5864,501,0,0,...,Credit Card,Home Improvement,25800,17.33,0.017,0.884,0.295,10.281513,6.218600,8.676758


In [19]:
## drop original columns and keep log columns to avoid multicolinearity
cols_to_drop = [
    'annual_income',
    'current_debt',
    'savings_assets'
]

X_train = X_train.drop(columns=cols_to_drop, axis =1)

In [20]:
out_info(X_train)

,col_name,method,num_outliers,%_outliers,uw,lw
0,derogatory_marks,IQR,5120,12.80,0.000000,0.000000
1,defaults_on_file,IQR,2163,5.41,0.000000,0.000000
2,delinquencies_last_2yrs,IQR,1419,3.55,2.500000,-1.500000
3,years_employed,IQR,1069,2.67,26.800000,-14.000000
4,credit_history_years,IQR,290,0.73,28.500000,-13.900000
5,debt_to_income_ratio,IQR,265,0.66,0.728500,-0.179500
6,current_debt_log,Z_Score,237,0.59,11.944860,6.438911
7,credit_score,Z_Score,104,0.26,837.731401,449.510949
8,age,Z_Score,87,0.22,68.359894,1.558606
9,annual_income_log,Z_Score,56,0.14,12.412784,8.876053


In [21]:
## same on X_test
X_test['annual_income_log'] = np.log1p(X_test['annual_income'])
X_test['current_debt_log'] = np.log1p(X_test['current_debt'])
X_test['savings_assets_log'] = np.log1p(X_test['savings_assets'])

In [22]:
cols_to_drop = [
    'annual_income',
    'current_debt',
    'savings_assets'
]

X_test = X_test.drop(columns=cols_to_drop)

In [23]:
X_train['derogatory_marks'].dtype

dtype('int64')

In [24]:
## encoding on categorical columns
X_train.columns

Index(['age', 'occupation_status', 'years_employed', 'credit_score',
       'credit_history_years', 'defaults_on_file', 'delinquencies_last_2yrs',
       'derogatory_marks', 'product_type', 'loan_intent', 'loan_amount',
       'interest_rate', 'debt_to_income_ratio', 'loan_to_income_ratio',
       'payment_to_income_ratio', 'annual_income_log', 'current_debt_log',
       'savings_assets_log'],
      dtype='object')

## one hot encoding

In [25]:
X_train_cat = X_train.select_dtypes(include ="object")
X_test_cat = X_test.select_dtypes(include ="object")

In [26]:
X_train_cat, X_test_cat

(      occupation_status    product_type       loan_intent
 39087          Employed     Credit Card          Personal
 30893           Student   Personal Loan         Education
 45278          Employed   Personal Loan          Personal
 16398          Employed  Line of Credit          Personal
 13653          Employed     Credit Card  Home Improvement
 ...                 ...             ...               ...
 11284           Student   Personal Loan          Personal
 44732           Student     Credit Card          Personal
 38158          Employed     Credit Card         Education
 860            Employed     Credit Card  Home Improvement
 15795          Employed  Line of Credit         Education
 
 [40000 rows x 3 columns],
       occupation_status    product_type loan_intent
 33553     Self-Employed  Line of Credit     Medical
 9427           Employed   Personal Loan   Education
 199            Employed     Credit Card    Business
 12447          Employed   Personal Loan    Busines

In [27]:
from sklearn.preprocessing import OneHotEncoder
ohe_enc = OneHotEncoder( drop = "first",
    handle_unknown="infrequent_if_exist",
    sparse_output= False
)

In [28]:
## insert this data into dataframe
pd.DataFrame(ohe_enc.fit_transform(X_train_cat),
           columns = ohe_enc.get_feature_names_out(),
           index=X_train.index)

,occupation_status_Self-Employed,occupation_status_Student,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
39087,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
30893,0.0,1.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
45278,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
16398,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
13653,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...
11284,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0
44732,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
38158,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
860,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0


In [29]:
## direct method to insert data into Data frame
X_train[ohe_enc.get_feature_names_out()] = ohe_enc.fit_transform(X_train_cat)
X_test[ohe_enc.get_feature_names_out()] = ohe_enc.transform(X_test_cat)

In [30]:
cat_cols = X_train_cat.columns
cat_cols

Index(['occupation_status', 'product_type', 'loan_intent'], dtype='object')

In [31]:
X_train.drop(columns= cat_cols, inplace= True)
X_test.drop(columns= cat_cols, inplace= True)

In [32]:
X_train.head()
X_test.head()

,age,years_employed,credit_score,credit_history_years,defaults_on_file,delinquencies_last_2yrs,derogatory_marks,loan_amount,interest_rate,debt_to_income_ratio,...,savings_assets_log,occupation_status_Self-Employed,occupation_status_Student,product_type_Line of Credit,product_type_Personal Loan,loan_intent_Debt Consolidation,loan_intent_Education,loan_intent_Home Improvement,loan_intent_Medical,loan_intent_Personal
33553,30,5.7,679,11.0,0,2,0,9600,10.72,0.200,...,9.035749,1.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0
9427,27,4.3,613,1.6,0,0,0,36700,13.55,0.180,...,5.805135,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0
199,21,1.0,590,1.5,0,1,3,60900,21.08,0.400,...,5.451038,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
12447,20,1.2,598,0.5,0,0,1,55300,16.17,0.143,...,2.564949,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
39489,49,20.9,725,8.9,0,0,0,32400,9.68,0.162,...,5.468060,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0


In [33]:
y_test.head()

33553    1
9427     1
199      0
12447    0
39489    1
Name: loan_status, dtype: int64

In [34]:
X_train_export = X_train.copy()
X_train_export["loan_status"] = y_train

X_test_export = X_test.copy()
X_test_export["loan_status"] = y_test

In [35]:
X_train_export.to_csv("X_train_clean.csv", index= False)
X_test_export.to_csv("X_test_clean.csv", index= False)

In [36]:
import joblib
joblib.dump(ohe_enc, "ohe_enc.joblib")

['ohe_enc.joblib']